In [ ]:
#!/usr/bin/env python3
"""
NMF Niche Analysis — All 41 Visium Samples
===========================================
Runs NMF on concatenated Cell2Location abundances across all samples,
computes CT/IM/Stroma regions, and generates cross-sample comparisons.

Usage: Run cells sequentially in Jupyter or as a script.
"""

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.decomposition import NMF
from sklearn.neighbors import NearestNeighbors
import os
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

META_PATH = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"
SAMPLE_DIR = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"
OUTPUT_DIR = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Niche_Analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Abundance key to use (q05 = 5% quantile, as in the IPF paper)
ABUNDANCE_KEY = 'q05_cell_abundance_w_sf'

# Region definition
TUMOR_LABELS = ['cancer_cancer']
STROMA_LABELS = ['benign_stroma', 'benign_fat', 'inter', 'stroma', 'benign_muscularis']
BOUNDARY_ROWS = 5  # 500µm ≈ 5 Visium spot spacings

# NMF parameters
K_RANGE = range(3, 21)       # range for K sweep
K_FINAL = 10                  # final K (adjust after seeing elbow)
N_RESTARTS = 5                # restarts for stability
RANDOM_STATE = 42

# Colors
REGION_COLORS = {'CT': '#E64B35', 'IM': '#F39B7F', 'Stroma': '#4DBBD5', 'Other': '#CCCCCC'}
VIVID_COLORS = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#aaffc3', '#dcbeff', '#9A6324', '#800000', '#469990'
]
HISTOLOGY_COLORS = {
    'cancer_cancer': '#E64B35', 'benign_stroma': '#4DBBD5',
    'benign_fat': '#F9E076', 'inter': '#91D1C2', 'stroma': '#8491B4',
    'benign_muscularis': '#B09C85', 'Unannotated': '#999999',
}

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def get_mets_group(row):
    """Assign metastasis group from metadata."""
    dist = str(row.get('Distant_Mets', 'False')).upper()
    ln = str(row.get('ln_only', 'False')).upper()
    if dist in ['TRUE', 'T']:
        return 'Distant_Mets'
    elif ln in ['TRUE', 'T']:
        return 'LN_Mets'
    return 'No_Mets'


def compute_regions(coords, histology, tumor_labels, stroma_labels, boundary_rows=5):
    """Compute CT/IM/Stroma regions using boundary distance method."""
    is_tumor = np.isin(histology, tumor_labels)
    is_stroma = np.isin(histology, stroma_labels)

    # Spot spacing
    nn = NearestNeighbors(n_neighbors=2).fit(coords)
    dists, _ = nn.kneighbors(coords)
    spot_spacing = np.median(dists[:, 1])
    boundary_dist = boundary_rows * spot_spacing

    tumor_coords = coords[is_tumor]
    stroma_coords = coords[is_stroma]

    region = np.full(len(histology), 'Other', dtype=object)

    if len(tumor_coords) > 0 and len(stroma_coords) > 0:
        nn_tumor = NearestNeighbors(n_neighbors=1).fit(tumor_coords)
        dist_to_tumor, _ = nn_tumor.kneighbors(coords)
        dist_to_tumor = dist_to_tumor.ravel()

        nn_stroma = NearestNeighbors(n_neighbors=1).fit(stroma_coords)
        dist_to_stroma, _ = nn_stroma.kneighbors(coords)
        dist_to_stroma = dist_to_stroma.ravel()

        for i in range(len(histology)):
            if is_tumor[i]:
                region[i] = 'IM' if dist_to_stroma[i] <= boundary_dist else 'CT'
            elif is_stroma[i]:
                region[i] = 'IM' if dist_to_tumor[i] <= boundary_dist else 'Stroma'
    elif len(tumor_coords) > 0:
        region[is_tumor] = 'CT'
    elif len(stroma_coords) > 0:
        region[is_stroma] = 'Stroma'

    return region


# ==============================================================================
# 3. LOAD ALL SAMPLES & CONCATENATE ABUNDANCES
# ==============================================================================

print("=" * 70)
print("STEP 1: Loading all samples")
print("=" * 70)

meta = pd.read_csv(META_PATH)
meta['deident'] = meta['deident'].astype(str)
meta['mets_group'] = meta.apply(get_mets_group, axis=1)

all_abundances = []
all_coords = []
all_histology = []
all_regions = []
all_sample_ids = []
all_mets = []
sample_spot_counts = {}

for _, row in meta.iterrows():
    sid = str(row['deident'])
    path = os.path.join(SAMPLE_DIR, f"{sid}.h5ad")
    if not os.path.exists(path):
        continue

    try:
        adata = sc.read_h5ad(path)

        if ABUNDANCE_KEY not in adata.obsm:
            print(f"  {sid}: missing {ABUNDANCE_KEY}, skipping")
            continue
        if 'histology' not in adata.obs.columns:
            print(f"  {sid}: missing histology, skipping")
            continue

        abund = adata.obsm[ABUNDANCE_KEY]
        coords = adata.obsm['spatial']
        histology = adata.obs['histology'].values

        # Compute regions
        region = compute_regions(coords, histology, TUMOR_LABELS, STROMA_LABELS, BOUNDARY_ROWS)

        n_spots = len(histology)
        all_abundances.append(abund)
        all_coords.append(coords)
        all_histology.extend(histology)
        all_regions.extend(region)
        all_sample_ids.extend([sid] * n_spots)
        all_mets.extend([row['mets_group']] * n_spots)
        sample_spot_counts[sid] = n_spots

        n_ct = np.sum(region == 'CT')
        n_im = np.sum(region == 'IM')
        n_str = np.sum(region == 'Stroma')
        print(f"  {sid}: {n_spots} spots (CT={n_ct}, IM={n_im}, Stroma={n_str}) — {row['mets_group']}")

    except Exception as e:
        print(f"  {sid}: ERROR — {e}")

# Concatenate
print(f"\nConcatenating {len(sample_spot_counts)} samples...")
X_combined = pd.concat(all_abundances, ignore_index=True)
ct_names = X_combined.columns.tolist()
clean_names = [c.replace("meanscell_abundance_w_sf_", "").replace("q05_cell_abundance_w_sf_", "") for c in ct_names]
X = X_combined.values

spot_meta = pd.DataFrame({
    'sample_id': all_sample_ids,
    'mets_group': all_mets,
    'histology': all_histology,
    'region': all_regions,
})

print(f"Combined matrix: {X.shape[0]} spots × {X.shape[1]} cell types")
print(f"Samples loaded: {len(sample_spot_counts)}")
print(f"\nMets distribution:")
print(spot_meta.groupby(['sample_id', 'mets_group']).size().reset_index().groupby('mets_group')['sample_id'].nunique())

# ==============================================================================
# 4. K SWEEP (ELBOW PLOT)
# ==============================================================================

print("\n" + "=" * 70)
print("STEP 2: NMF K sweep")
print("=" * 70)

errors = []
for k in K_RANGE:
    m = NMF(n_components=k, init='nndsvda', random_state=RANDOM_STATE, max_iter=1000)
    m.fit_transform(X)
    errors.append(m.reconstruction_err_)
    print(f"  K={k}: error={m.reconstruction_err_:.2f}, iters={m.n_iter_}")

fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
ax.plot(list(K_RANGE), errors, 'o-', color='#4363d8', linewidth=2, markersize=6)
ax.set_xlabel('Number of niches (K)', fontsize=13)
ax.set_ylabel('Reconstruction error', fontsize=13)
ax.set_title('NMF K Selection — All 41 Samples', fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'nmf_elbow_all_samples.png'), dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 5. FIT FINAL NMF MODEL
# ==============================================================================

print(f"\n{'=' * 70}")
print(f"STEP 3: Fitting final NMF with K={K_FINAL}")
print("=" * 70)

model = NMF(n_components=K_FINAL, init='nndsvda', random_state=RANDOM_STATE, max_iter=2000, tol=1e-4)
W = model.fit_transform(X)  # spots × K
H = model.components_        # K × cell_types

print(f"Reconstruction error: {model.reconstruction_err_:.2f}")
print(f"Iterations: {model.n_iter_}")
print(f"W: {W.shape}, H: {H.shape}")

# Add niche info to spot metadata
spot_meta['dominant_niche'] = np.argmax(W, axis=1) + 1
for k in range(K_FINAL):
    spot_meta[f'niche_{k+1}_loading'] = W[:, k]

# Print niche compositions
print("\n" + "=" * 60)
print("NICHE COMPOSITIONS (Top 5 cell types)")
print("=" * 60)
for i in range(K_FINAL):
    top_idx = np.argsort(H[i])[::-1][:5]
    print(f"\nNiche {i+1}:")
    for j in top_idx:
        print(f"  {clean_names[j]}: {H[i, j]:.4f}")

# ==============================================================================
# 6. GLOBAL HEATMAP — Cell type contributions per niche
# ==============================================================================

print(f"\n{'=' * 70}")
print("STEP 4: Generating plots")
print("=" * 70)

H_norm = H / H.sum(axis=1, keepdims=True)
ct_df = pd.DataFrame(H_norm.T, index=clean_names, columns=[f"Niche {i+1}" for i in range(K_FINAL)])

# Collect top 5 per niche
top_cts = []
for i in range(K_FINAL):
    top_idx = np.argsort(H[i])[::-1][:5]
    for j in top_idx:
        if clean_names[j] not in top_cts:
            top_cts.append(clean_names[j])

ct_df_top = ct_df.loc[top_cts].copy()

g = sns.clustermap(
    ct_df_top, cmap='YlOrRd',
    figsize=(14, max(8, len(top_cts) * 0.4)),
    row_cluster=True, col_cluster=False,
    linewidths=0.5, linecolor='white',
    dendrogram_ratio=(0.1, 0.0),
    cbar_kws={'label': 'Relative contribution', 'shrink': 0.5},
    yticklabels=True, xticklabels=True,
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=11, rotation=45, ha='right')
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=9)
g.fig.suptitle(f'NMF Niche Cell Type Composition (K={K_FINAL}, All Samples)',
               fontsize=14, fontweight='bold', y=1.02)
plt.savefig(os.path.join(OUTPUT_DIR, 'niche_heatmap_all_samples.png'), dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 7. NICHE × REGION CROSS-TABULATION
# ==============================================================================

cross_tab = pd.crosstab(spot_meta['dominant_niche'], spot_meta['region'], normalize='index') * 100
col_order = [c for c in ['CT', 'IM', 'Stroma', 'Other'] if c in cross_tab.columns]
cross_tab = cross_tab[col_order]

fig, ax = plt.subplots(figsize=(12, 6), facecolor='white')
cross_tab.plot(kind='bar', stacked=True, ax=ax,
               color=[REGION_COLORS[c] for c in col_order],
               edgecolor='white', linewidth=0.5)
ax.set_xlabel('Niche', fontsize=13)
ax.set_ylabel('% of spots', fontsize=13)
ax.set_title(f'Niche × Region (All Samples, K={K_FINAL})', fontsize=15, fontweight='bold')
ax.legend(title='Region', bbox_to_anchor=(1.02, 1), fontsize=11)
ax.set_xticklabels([f'Niche {i}' for i in cross_tab.index], rotation=0, fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'niche_region_crosstab_all.png'), dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 8. NICHE PROPORTIONS BY METS GROUP (SAMPLE-LEVEL)
# ==============================================================================

# Compute sample-level niche proportions
sample_niche = spot_meta.groupby(['sample_id', 'mets_group', 'dominant_niche']).size().reset_index(name='count')
sample_totals = spot_meta.groupby('sample_id').size().reset_index(name='total')
sample_niche = sample_niche.merge(sample_totals, on='sample_id')
sample_niche['proportion'] = sample_niche['count'] / sample_niche['total'] * 100

# Boxplot: niche proportions across mets groups
mets_order = ['No_Mets', 'LN_Mets', 'Distant_Mets']
mets_palette = {'No_Mets': '#3cb44b', 'LN_Mets': '#ffe119', 'Distant_Mets': '#e6194b'}

fig, axes = plt.subplots(2, 5, figsize=(28, 10), facecolor='white')
for i, ax in enumerate(axes.flat):
    niche_num = i + 1
    if niche_num > K_FINAL:
        ax.axis('off')
        continue

    data = sample_niche[sample_niche['dominant_niche'] == niche_num]
    # Ensure all mets groups present
    for mg in mets_order:
        if mg not in data['mets_group'].values:
            pass

    sns.boxplot(data=data, x='mets_group', y='proportion', order=mets_order,
                palette=mets_palette, ax=ax, fliersize=3, width=0.6)
    sns.stripplot(data=data, x='mets_group', y='proportion', order=mets_order,
                  color='black', alpha=0.5, size=4, ax=ax)

    # Get top 2 cell types for title
    top2 = np.argsort(H[i])[::-1][:2]
    subtitle = " + ".join([clean_names[j].split("(")[0].strip()[:20] for j in top2])
    ax.set_title(f'Niche {niche_num}\n{subtitle}', fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('% spots' if i % 5 == 0 else '')
    ax.tick_params(axis='x', rotation=30, labelsize=8)

plt.suptitle(f'Niche Proportions by Metastasis Group (K={K_FINAL})',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'niche_proportions_by_mets.png'), dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 9. NICHE PROPORTIONS BY REGION × METS GROUP
# ==============================================================================

# For each sample × region, compute niche proportions
sample_region_niche = spot_meta.groupby(
    ['sample_id', 'mets_group', 'region', 'dominant_niche']
).size().reset_index(name='count')
sample_region_totals = spot_meta.groupby(['sample_id', 'region']).size().reset_index(name='total')
sample_region_niche = sample_region_niche.merge(sample_region_totals, on=['sample_id', 'region'])
sample_region_niche['proportion'] = sample_region_niche['count'] / sample_region_niche['total'] * 100

# Stacked bar: mean niche proportions per region × mets
for region_name in ['CT', 'IM', 'Stroma']:
    rdata = sample_region_niche[sample_region_niche['region'] == region_name]
    if len(rdata) == 0:
        continue

    pivot = rdata.pivot_table(index='mets_group', columns='dominant_niche',
                               values='proportion', aggfunc='mean').fillna(0)
    pivot = pivot.reindex(mets_order)

    fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
    pivot.plot(kind='bar', stacked=True, ax=ax,
               color=VIVID_COLORS[:K_FINAL], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{region_name} — Mean Niche Composition by Mets Group', fontsize=14, fontweight='bold')
    ax.set_ylabel('Mean % of spots', fontsize=12)
    ax.set_xlabel('')
    ax.legend(title='Niche', bbox_to_anchor=(1.02, 1), fontsize=9,
              labels=[f'N{i}' for i in pivot.columns])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'niche_composition_{region_name}_by_mets.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# ==============================================================================
# 10. PER-SAMPLE SPATIAL MAPS (SELECT EXAMPLES)
# ==============================================================================

# Pick representative samples: one per mets group with balanced histology
example_samples = {
    'No_Mets': '38',     # 46.8% cancer
    'LN_Mets': '107',    # 48.3% cancer
    'Distant_Mets': '80' # 53.9% cancer
}

# Reconstruct per-sample data from spot_meta
cumulative = 0
sample_slices = {}
for sid, n_spots in sample_spot_counts.items():
    sample_slices[sid] = (cumulative, cumulative + n_spots)
    cumulative += n_spots

for mets_label, sid in example_samples.items():
    if sid not in sample_slices:
        print(f"  Sample {sid} not found, skipping")
        continue

    s, e = sample_slices[sid]
    W_sample = W[s:e]
    coords_sample = np.concatenate(all_coords)[s:e] if isinstance(all_coords[0], np.ndarray) else None
    hist_sample = np.array(all_histology[s:e])
    region_sample = np.array(all_regions[s:e])
    dominant_sample = spot_meta.iloc[s:e]['dominant_niche'].values

    # Reload coords from file (safer)
    adata_tmp = sc.read_h5ad(os.path.join(SAMPLE_DIR, f"{sid}.h5ad"))
    coords_sample = adata_tmp.obsm['spatial']
    y_sample = coords_sample[:, 1].max() - coords_sample[:, 1]

    fig, axes = plt.subplots(1, 3, figsize=(30, 9), facecolor='white')

    # Panel 1: Histology
    ax = axes[0]
    for h_label in np.unique(hist_sample):
        mask = hist_sample == h_label
        color = HISTOLOGY_COLORS.get(h_label, '#999999')
        ax.scatter(coords_sample[mask, 0], y_sample[mask], c=color, s=8,
                   edgecolors='none', label=h_label)
    ax.set_title('Pathologist Histology', fontsize=16, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.legend(loc='center left', bbox_to_anchor=(0, -0.1), fontsize=9, markerscale=3, ncol=2)

    # Panel 2: CT/IM/Stroma
    ax = axes[1]
    for r_label in ['CT', 'IM', 'Stroma', 'Other']:
        mask = region_sample == r_label
        if mask.sum() > 0:
            ax.scatter(coords_sample[mask, 0], y_sample[mask],
                       c=REGION_COLORS[r_label], s=8, edgecolors='none', label=r_label)
    ax.set_title('CT / IM / Stroma', fontsize=16, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.legend(loc='center left', bbox_to_anchor=(0, -0.1), fontsize=9, markerscale=3, ncol=2)

    # Panel 3: Dominant Niche
    ax = axes[2]
    for n_i in range(K_FINAL):
        mask = dominant_sample == (n_i + 1)
        if mask.sum() > 0:
            ax.scatter(coords_sample[mask, 0], y_sample[mask],
                       c=VIVID_COLORS[n_i], s=8, edgecolors='none', label=f'Niche {n_i+1}')
    ax.set_title(f'NMF Dominant Niche (K={K_FINAL})', fontsize=16, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.legend(loc='center left', bbox_to_anchor=(0, -0.1), fontsize=9, markerscale=3, ncol=3)

    plt.suptitle(f'Sample {sid} — {mets_label.replace("_", " ")}',
                 fontsize=20, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'three_panel_{sid}_{mets_label}.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Per-sample niche loading maps (black background)
    fig, axes_hot = plt.subplots(2, 5, figsize=(35, 14), facecolor='black')
    fig.subplots_adjust(hspace=0.3, wspace=0.15)
    for i, ax in enumerate(axes_hot.flat):
        if i >= K_FINAL:
            ax.axis('off')
            continue
        ax.set_facecolor('black')
        vals = W_sample[:, i]
        vmax = np.percentile(vals, 98) if np.percentile(vals, 98) > 0 else 1

        sc_plot = ax.scatter(coords_sample[:, 0], y_sample, c=vals, cmap='hot',
                             s=10, edgecolors='none', vmin=0, vmax=vmax)
        top2 = np.argsort(H[i])[::-1][:2]
        subtitle = " + ".join([clean_names[j].split("(")[0].strip()[:20] for j in top2])
        ax.set_title(f'Niche {i+1}\n{subtitle}', fontsize=11, fontweight='bold', color='white')
        ax.set_aspect('equal')
        ax.axis('off')
        cbar = plt.colorbar(sc_plot, ax=ax, shrink=0.6, pad=0.02)
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

    plt.suptitle(f'Sample {sid} ({mets_label.replace("_", " ")}) — Niche Loadings',
                 fontsize=16, fontweight='bold', color='white', y=1.02)
    plt.savefig(os.path.join(OUTPUT_DIR, f'niche_loadings_{sid}_{mets_label}.png'),
                dpi=300, bbox_inches='tight', facecolor='black')
    plt.show()

# ==============================================================================
# 11. SAVE RESULTS
# ==============================================================================

print(f"\n{'=' * 70}")
print("STEP 5: Saving results")
print("=" * 70)

# Save spot-level metadata with niche assignments
spot_meta.to_csv(os.path.join(OUTPUT_DIR, 'spot_niche_assignments.csv'), index=False)
print(f"  Saved spot_niche_assignments.csv ({len(spot_meta)} spots)")

# Save niche compositions (H matrix)
H_df = pd.DataFrame(H, index=[f"Niche_{i+1}" for i in range(K_FINAL)], columns=clean_names)
H_df.to_csv(os.path.join(OUTPUT_DIR, 'niche_cell_type_weights.csv'))
print(f"  Saved niche_cell_type_weights.csv")

# Save sample-level niche proportions
sample_niche_prop = spot_meta.groupby(['sample_id', 'mets_group', 'dominant_niche']).size().unstack(fill_value=0)
sample_niche_prop = sample_niche_prop.div(sample_niche_prop.sum(axis=1), axis=0) * 100
sample_niche_prop.columns = [f'Niche_{c}_pct' for c in sample_niche_prop.columns]
sample_niche_prop.to_csv(os.path.join(OUTPUT_DIR, 'sample_niche_proportions.csv'))
print(f"  Saved sample_niche_proportions.csv")

# Save sample × region × niche proportions
sample_region_niche.to_csv(os.path.join(OUTPUT_DIR, 'sample_region_niche_proportions.csv'), index=False)
print(f"  Saved sample_region_niche_proportions.csv")

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print("Done!")